# **HPLC Pigments: EOFs**

Author: Sasha Kramer | GitHub: https://github.com/sashajane19/HPLCcluster | Reference: Kramer and Siegel, 2019: https://doi.org/10.1029/2019JC015604

Empirical Orthogonal Function (EOF) analysis of HPLC pigment data normalized to chlorophyll a. Pigments are mean-centred and standardized before PCA. EOF loadings and amplitude functions (AFs) are plotted as bar charts and on a global map respectively. Pearson correlations between AFs and pigment ratios are annotated on the loading plots.

**Overlap note:** Data loading, QC, pigment selection, and Tchla normalization are shared with HPLC_cluster.ipynb. If you are running both notebooks in the same session or kernel, you can skip those cells here and start after normalization.

### Import libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib
import scipy.io
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
import warnings
import os
import cartopy.crs as ccrs
import cartopy.feature as cfeature
warnings.filterwarnings('ignore')

### Import your data.

Here we are using a standard pigment file that corresponds to the data in Kramer et al. (2022) - https://doi.org/10.1594/PANGAEA.937536. You can also use your own pigment file, but check if the header corresponds to the one used below.

In [ ]:
pigment_cols = ['Tchla','Tchlb','Tchlc','ABcaro','ButFuco','HexFuco','Allo','Diadino','Diato','Fuco','Perid','Zea','MVChla','DVchla','Chllide','MVChlb','DVchlb','Chlc12','Chlc3','Lut','Neo','Viola','Phytin','Phide','Pras']
os.chdir('/Users/skramer/Documents/UCSB/Research/Data/HPLC_Aph_Rrs')
mat = scipy.io.loadmat('Global_Rrs_HPLC_QC_EXP.mat')
Global_RHPLC = pd.DataFrame(mat['Global_RHPLC'], columns=pigment_cols)
Global_Rlat   = mat['Global_Rlat'].flatten()    # latitude vector  (n_samples,)
Global_Rlon   = mat['Global_Rlon'].flatten()    # longitude vector (n_samples,)

In [ ]:
pigment_cols = ['Tchla','Tchlb','Tchlc','ABcaro','ButFuco','HexFuco','Allo','Diadino','Diato','Fuco','Perid','Zea','MVChla','DVchla','Chllide','MVChlb','DVchlb','Chlc12','Chlc3','Lut','Neo','Viola','Phytin','Phide','Pras']

#os.chdir('/Users/Research/Data/HPLC') #insert your path to the HPLC data here

Global_RHPLC = pd.read_csv('Global_HPLC_all.csv', header=0)
Global_RHPLC.columns = pigment_cols
Global_Rlat  = pd.read_csv('Global_lat.csv').values.flatten() # this is assuming you have separate lat/lon info - you might index this from your file directly
Global_Rlon  = pd.read_csv('Global_lon.csv').values.flatten()

### Quality Control

Values below the NASA GSFC detection limit are set to zero.

A generic threshold of 0.001 mg/m3 applies to: Chlc3, Chlc12, Chllide, Viola, Diadino, Diato, Allo, Zea, Lut, and ABcaro.

Pigment-specific thresholds are applied for the remaining pigments in the code cell below, and then pigments where > 80% of values fall below detection are candidates for removal. The denominator below reflects the number of smaples (here, 145, but change for your dataset).

In [ ]:
detection_limits = {
'Phide':   0.003,   
'Perid':   0.003,   
'MVChlb':  0.003,   
'Phytin':  0.003,   
'ButFuco': 0.002,   
'Fuco':    0.002,   
'Neo':     0.002,   
'Pras':    0.002,   
'HexFuco': 0.002,   
'DVchla':  0.002,
}
Global_RHPLC = Global_RHPLC.copy()  # work on a copy; preserves the original loaded data
for pigment, limit in detection_limits.items():
    mask = (Global_RHPLC[pigment] != 0) & (Global_RHPLC[pigment] <= limit)
n_flagged = mask.sum()
Global_RHPLC.loc[mask, pigment] = 0
print(f"  {pigment:>8s}: {n_flagged:3d} value(s) <= {limit} mg/m3 set to 0")

In [ ]:
n_samples = len(Global_RHPLC)
below_detection = {
col: 100 * (Global_RHPLC[col] <= 0.001).sum() / n_samples
for col in pigment_cols
}
below_det_df = (
pd.DataFrame
.from_dict(below_detection, orient='index', columns=['% Below Detection'])
.sort_values('% Below Detection', ascending=False)
)
print(below_det_df.to_string())
print('nPigments with > 80% of values below detection (candidates for removal):')
candidates = below_det_df[below_det_df['% Below Detection'] > 80]
print(candidates if not candidates.empty else '  None found')

### Pigment selection for statistical analysis

Step 1: Remove degradation products — Chllide, Phytin, Phide.

Step 2: Remove redundant pigments (total or summed indices that duplicate information) and any pigments with more than 80% of values below detection in your dataset. In the reference dataset (n = 145), Lut, Pras, and DVchlb were also removed at this step. Adjust the list in Cell 7 based on your own below-detection analysis in Cell 5.

In [ ]:
deg_pigments = ['Chllide', 'Phytin', 'Phide']
Rpigcluster1 = Global_RHPLC.drop(columns=deg_pigments)
print(f"After removing degradation products: {Rpigcluster1.shape[1]} pigments remaining")
print(list(Rpigcluster1.columns))

In [ ]:
redundant_pigments = [
'Tchlb', 'Tchlc', 'ABcaro',      # redundant total / summed pigments
'Diadino', 'Diato',               # > 80% below detection (reference dataset)
'MVChla',                         # redundant with Tchla
'DVchlb', 'Lut', 'Pras',          # > 80% below detection (reference dataset)
]
Rpigcluster2 = Rpigcluster1.drop(columns=redundant_pigments)
label2 = list(Rpigcluster2.columns)
print(f"Pigments retained for clustering ({len(label2)}): {label2}")

In [ ]:
normlabel = [p for p in label2 if p != 'Tchla']   # 12 accessory pigments
normchl = (
Rpigcluster2[normlabel].values
/ Global_RHPLC['Tchla'].values[:, np.newaxis]
)
normchl_df = pd.DataFrame(normchl, columns=normlabel)
print(f"Normalized matrix: {normchl_df.shape[0]} samples x {normchl_df.shape[1]} pigments")
normchl_df.head()

### Define a plot color vector.

In [ ]:
cmap_jet = plt.cm.jet
colors = [cmap_jet(i / 4) for i in range(5)]
fig, ax = plt.subplots(figsize=(5, 1))
for i, c in enumerate(colors):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=c))
ax.set_xlim(0, 5); ax.set_ylim(0, 1)
ax.set_xticks(np.arange(0.5, 5.5)); ax.set_xticklabels([f'C{i+1}' for i in range(5)])
ax.set_yticks([]); ax.set_title('Cluster colors (jet, n=5)')
plt.tight_layout(); plt.show()

### Pigment reordering for EOF analysis

Pigments are reordered so that members of the same hierarchical cluster from the dendrogram analysis sit adjacent to each other in the loading bar charts. Update the eof_order list if your cluster results differ from the reference dataset.

In [ ]:
eof_order = [7, 10, 11, 2, 0, 1, 3, 8, 9, 4, 5, 6] # Update this order to match your cluster dendrogram results
eof_cols = [normlabel[i] for i in eof_order]
pigEOF = normchl_df[eof_cols].values    # shape: (n_samples, 12)
pigEOF_df = pd.DataFrame(pigEOF, columns=eof_cols) # These labels must match eof_order — update if you change the order above

plotlabel = ['MVChlb','Neo','Viola','Allo','ButFuco','HexFuco',
'Fuco','Chlc12','Chlc3','Perid','Zea','DVchla']
print(f"EOF input matrix: {pigEOF_df.shape}")
print(f"Pigment order  : {plotlabel}")

### Mean centering and standardizing (z-scoring) the pigment values

Each pigment ratio is z-score normalized (subtract mean, divide by standard deviation) before PCA. NaN values are excluded from mean/std calculations (np.nanmean / np.nanstd), equivalent to MATLAB's nanmean and nanstd. 

In [ ]:
pigmeans = np.nanmean(pigEOF, axis=0)   # shape: (12,)
pigstd   = np.nanstd(pigEOF,  axis=0)   # shape: (12,)
pigs_center = (pigEOF - pigmeans) / pigstd   # shape: (n_samples, 12)
print(f"Z-scored matrix  : {pigs_center.shape}")
print(f"Column means (should be ~0): {np.nanmean(pigs_center, axis=0).round(6)}")
print(f"Column stds  (should be ~1): {np.nanstd(pigs_center,  axis=0).round(6)}")

### EOF calculation (as PCA)

Calculate principal components and look at variance explained by each component.

In [ ]:
complete_mask = ~np.any(np.isnan(pigs_center), axis=1)
n_complete = complete_mask.sum()
print(f"Complete rows (no NaN): {n_complete} / {len(pigs_center)}")
pigs_center_complete = pigs_center[complete_mask]
pca_model = PCA(n_components=pigs_center.shape[1])
pca_model.fit(pigs_center_complete)

EOFs_pig   = pca_model.components_.T 
eigvalues_pig = pca_model.explained_variance_

AFs_pig = pigs_center @ EOFs_pig
print(f"nEOFs_pig shape (loadings)         : {EOFs_pig.shape}")
print(f"AFs_pig  shape (amplitude funcs)  : {AFs_pig.shape}")
print(f"Eigenvalues                       : {eigvalues_pig.round(3)}")

var_explained = eigvalues_pig / eigvalues_pig.sum()
var_cutoff = 0.031   # retain modes explaining > 3.1% of variance; adjust as needed based on your results!

# Modes to keep
keep_mask = var_explained > var_cutoff
n_modes   = keep_mask.sum()
print(f"Number of modes retained (> {var_cutoff*100:.1f}% variance): {n_modes}")
expvar         = var_explained[keep_mask]
expvar_percent = expvar * 100

perc_exp = pd.DataFrame({
'Mode'               : np.arange(1, n_modes + 1),
'Var Explained (%)'  : expvar_percent.round(2),
'Cumulative Var (%)' : np.cumsum(expvar_percent).round(2)
})
print(perc_exp.to_string(index=False))

# Label strings for annotating plots 
expvar_str = [f"{v:.1f}%" for v in expvar_percent]
print(f"nMode labels: {expvar_str}")

### Trim matrix to retained modes (here, first 6)

Keeping only the modes that exceed the variance threshold.

In [ ]:
EOFs_pigC = EOFs_pig[:, keep_mask]    # (n_pigments × n_modes)
AFs_pigC  = AFs_pig[:, keep_mask]     # (n_samples  × n_modes)

# Attach lat/lon alongside amplitude functions for spatial analysis
latlon = np.column_stack([Global_Rlat, Global_Rlon])
AFs_pig_latlon = np.column_stack([latlon, AFs_pigC])

# Extract individual amplitude functions for convenience
AF1p = AFs_pigC[:, 0]
AF2p = AFs_pigC[:, 1]
AF3p = AFs_pigC[:, 2]
AF4p = AFs_pigC[:, 3]
AF5p = AFs_pigC[:, 4]
AF6p = AFs_pigC[:, 5]
print(f"EOFs_pigC : {EOFs_pigC.shape}  |  AFs_pigC : {AFs_pigC.shape}")

### Correlation analysis between AFs and lat/lon (to check geographic structure)

Two correlation matrices are computed:
- corrmat1 — AFs with pigments AND lat/lon (to check for geographic structure in each mode)
- corrmat — AFs with pigments only

You will also define a pairwise correlation helpful function to return R and P values.

In [ ]:
def pairwise_corrcoef(X):
    """
    Compute pairwise Pearson correlation matrix R and p-value matrix P
    for a 2-D array X (n_samples x n_variables), ignoring NaN pairs.
    Equivalent to MATLAB: [R, P] = corrcoef(X, 'rows', 'pairwise')
    Returns
    
    R : (n_variables, n_variables) correlation matrix
    P : (n_variables, n_variables) p-value matrix
    """
    n = X.shape[1]
    R = np.full((n, n), np.nan)
    P = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            # Pairwise complete: keep rows where BOTH columns are non-NaN
            mask_ij = ~(np.isnan(X[:, i]) | np.isnan(X[:, j]))
            if mask_ij.sum() > 2:
                r, p = pearsonr(X[mask_ij, i], X[mask_ij, j])
                R[i, j] = r
                P[i, j] = p
            if i == j:
                R[i, j] = 1.0
                P[i, j] = 0.0
    return R, P

In [ ]:
# corrmat1: AFs + pigments + lat/lon  (checking geographic structure)
corrmat1 = np.column_stack([AFs_pigC, pigEOF, latlon])
R_p1, P_p1 = pairwise_corrcoef(corrmat1)
print("corrmat1 (AFs + pigs + lat/lon) computed.")

# corrmat:  AFs + pigments only
corrmat = np.column_stack([AFs_pigC, pigEOF])
R_pig, P_pig = pairwise_corrcoef(corrmat)
print("corrmat  (AFs + pigs only)      computed.")

# Sanity check: off-diagonal AF correlations should be near 0
af_cross_corr = R_pig[:n_modes, :n_modes]
print(f"nAF cross-correlation matrix (should be identity):")
print(af_cross_corr.round(3))

R_pig_trim = R_pig[:n_modes, n_modes:]    # (n_modes x n_pigments)
P_pig_trim = P_pig[:n_modes, n_modes:]

# For the bar plot annotation: transpose to (n_pigments x n_modes)
R_rearr = R_pig_trim.T
print(f"nR_pig_trim shape : {R_pig_trim.shape}  (n_modes x n_pigs)")
print(f"R_rearr    shape : {R_rearr.shape}     (n_pigs  x n_modes)")

### EOF loading bar charts

Each bar represents the loading of one pigment on a given EOF mode. Bars are colored by cluster membership (matching the dendrogram color scheme). Pearson correlation coefficients (×100, integer) between the AF and each pigment ratio are annotated beneath each bar.

In [ ]:
def plot_eof_mode(mode_idx, EOFs_pigC, R_rearr, plotlabel, colors, expvar_str,
                  save=True):
    """
    Plot EOF loadings for one mode as a colored bar chart with correlation
    coefficient annotations.
    Parameters
    
    mode_idx   : int   — 0-based mode index (0 = Mode 1, etc.)
    EOFs_pigC  : ndarray (n_pigs x n_modes) — loading matrix
    R_rearr    : ndarray (n_pigs x n_modes) — pigment-AF correlation matrix
    plotlabel  : list[str]                  — pigment labels (ordered as plotted)
    colors     : list                       — cluster colors from Cell 10
    expvar_str : list[str]                  — variance strings, e.g. ['28.3%', ...]
    save       : bool                       — save figure to PNG
    """
    n_pigs = len(plotlabel)
    mode_num = mode_idx + 1   # human-readable 1-based mode number
    bar_groups = {
        'indices': [
            list(range(0, 4)),    # bars 1-4
            list(range(4, 6)),    # bars 5-6
            list(range(6, 9)),    # bars 7-9
            [9],                  # bar  10
            list(range(10, 12)),  # bars 11-12
        ],
        'colors': [
            colors[2],  
            colors[1],  
            colors[3],  
            colors[4],  
            colors[0],  
        ]
    }
    fig, ax = plt.subplots(figsize=(10, 6))
    x_pos = np.arange(1, n_pigs + 1)
    for idx_list, clr in zip(bar_groups['indices'], bar_groups['colors']):
        ax.bar(x_pos[idx_list],
               EOFs_pigC[idx_list, mode_idx],
               color=clr, edgecolor='none')
    ax.set_xlim(0, n_pigs + 1)
    ax.set_ylim(-1, 1)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(plotlabel, rotation=90, fontsize=24)
    ax.set_yticks(np.arange(-1, 1.5, 0.5))
    ax.set_ylabel('EOF Loading', fontsize=20)
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.tick_params(axis='both', labelsize=24)
    ax.text(0.3, 0.82, f'Mode {mode_num}: {expvar_str[mode_idx]}',
            transform=ax.transAxes,
            fontsize=26, fontweight='bold')
    for xi, ri in zip(x_pos, R_rearr[:, mode_idx]):
        if not np.isnan(ri):
            ax.text(xi - 0.25, -0.88,
                    f"{int(round(ri * 100)):d}",
                    fontsize=15, fontweight='bold')
    plt.tight_layout()
    if save:
        fname = f'eof_loading_mode{mode_num}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f"Saved: {fname}")
    plt.show()

In [ ]:
for mode_idx in range(n_modes):
    plot_eof_mode(
    mode_idx   = mode_idx,
    EOFs_pigC  = EOFs_pigC,
    R_rearr    = R_rearr,
    plotlabel  = plotlabel,
    colors     = colors,
    expvar_str = expvar_str,
    save       = False, # change if you want to save figures!
)

### Global map of amplitude functions

Amplitude functions (scores) for each EOF mode are plotted at sample locations using a diverging colormap centred on zero. 

In [ ]:
def plot_af_map(af_values, lat, lon, mode_num, expvar_str, vmin=-2, vmax=2,
                cmap='RdBu_r', save=True):
    """
    Plot amplitude function values on a global Cartopy map.
    Parameters
    ----------
    af_values : 1-D array
        amplitude function for one mode (n_samples,)
    lat, lon : 1-D arrays
        sample locations
    mode_num : int
        1-based mode number for title/label
    expvar_str : str
        variance string for this mode, e.g. '28.3%'
    vmin, vmax : float
        colorbar limits (equivalent to MATLAB caxis)
    cmap : str
        matplotlib colormap name; 'RdBu_r' approximates GMT_polar
    save : bool
        save figure to PNG
    """
    fig = plt.figure(figsize=(14, 7), facecolor='white')
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    ax.set_extent([-180, 180, -80, 80], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor=[0.8, 0.8, 0.8], zorder=1)
    ax.add_feature(cfeature.COASTLINE, linewidth=1.5, zorder=2)
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                      linewidth=0.5, color='gray', alpha=0.5,
                      linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 14, 'weight': 'bold'}
    gl.ylabel_style = {'size': 14, 'weight': 'bold'}
    sc = ax.scatter(lon, lat, c=af_values, s=70,
                    cmap=cmap, vmin=vmin, vmax=vmax,
                    transform=ccrs.PlateCarree(), zorder=3)
    cb = plt.colorbar(sc, ax=ax, orientation='vertical',
                      fraction=0.025, pad=0.04)
    cb.set_label(f'AF{mode_num} magnitude', fontsize=18)
    cb.ax.tick_params(labelsize=16)
    ax.set_title(f'Mode {mode_num}  ({expvar_str})',
                 fontsize=18, fontweight='bold')
    plt.tight_layout()
    if save:
        fname = f'af_map_mode{mode_num}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f"Saved: {fname}")
    plt.show()

In [ ]:
for mode_idx in range(n_modes):
    plot_af_map(
    af_values  = AFs_pigC[:, mode_idx],
    lat        = Global_Rlat,
    lon        = Global_Rlon,
    mode_num   = mode_idx + 1,
    expvar_str = expvar_str[mode_idx],
    vmin       = -2,
    vmax       =  2,
    cmap       = 'RdBu_r',   # replace with cmocean.cm.balance for closer GMT_polar match
    save       = False, # change if you want to save figures!
)

### Export results for further analysis.

In [ ]:
af_cols = [f'AF{i+1}' for i in range(n_modes)]
af_df = pd.DataFrame(AFs_pigC, columns=af_cols)
af_df.insert(0, 'Latitude',  Global_Rlat)
af_df.insert(1, 'Longitude', Global_Rlon)
af_df.to_csv('amplitude_functions.csv', index=False)
print("Saved: amplitude_functions.csv")

eof_df = pd.DataFrame(EOFs_pigC, index=plotlabel, columns=af_cols)
eof_df.to_csv('eof_loadings.csv')
print("Saved: eof_loadings.csv")

perc_exp.to_csv('variance_explained.csv', index=False)
print("Saved: variance_explained.csv")

R_trim_df = pd.DataFrame(R_pig_trim, columns=plotlabel,
index=[f'AF{i+1}' for i in range(n_modes)])
R_trim_df.to_csv('af_pigment_correlations.csv')
print("Saved: af_pigment_correlations.csv")